In [1]:
import argparse
import itertools
import os
import pathlib
import sys
from functools import reduce

import duckdb
import pandas as pd
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm
profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir  # default to root_dir instead of NAS

In [2]:
patient_id_file = pathlib.Path(f"{profile_base_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patient_id_file, header=None, names=["patient_id"]
).patient_id.tolist()

In [3]:
out_dict = {
    "file_path": [],
    "patient_id": [],
    "well_fov": [],
    "feature_type": [],
    "compartment": [],
    # "df_shape": [],
}

# get all well_fovs for a patient
for patient in tqdm.tqdm(patients, desc="Processing patients", leave=True):
    patient_dir = profile_base_dir / "data" / patient / "extracted_features"
    well_fovs = patient_dir.glob("*")  # get all well_fovs for a patient
    # print(f"Found well_fovs: {well_fovs}")
    for well_fov in tqdm.tqdm(well_fovs, desc="Processing well_fovs", leave=False):
        if "stats" in well_fov.stem:
            continue
        features = pathlib.Path(well_fov).glob("*.parquet")
        for feature in features:
            feature_type = feature.stem.split("_")[2]
            compartment = feature.stem.split("_")[0]
            out_dict["file_path"].append(feature)
            out_dict["patient_id"].append(patient)
            out_dict["well_fov"].append(feature.parent.stem)
            out_dict["feature_type"].append(feature_type)
            out_dict["compartment"].append(compartment)
            # out_dict["df_shape"].append(pd.read_parquet(feature).shape)
df = pd.DataFrame(out_dict)
# df = df.loc[df["patient_id"] == "NF0014_T1"]
df = df.loc[(df["feature_type"] == "Granularity") & (df["compartment"] != "Organoid")]
df = df.loc[(df["compartment"] != "Organoid")]

df.head()

Processing patients:   0%|          | 0/13 [00:00<?, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

...

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

Processing well_fovs: 0it [00:00, ?it/s]

,file_path,patient_id,well_fov,feature_type,compartment
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
5,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm
16,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cytoplasm
23,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell
27,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,G8-1,Granularity,Cell


In [4]:
from tqdm import tqdm

tqdm.pandas()


def safe_read_shape(x):
    try:
        df = pd.read_parquet(x)
        return df.shape, df.isna().sum().sum()
    except Exception as e:
        print(f"Error reading {x}: {e}")
        return None, None


# if not pathlib.Path("../logs/feature_file_info.parquet").exists():
df[["df_shape", "missing_values"]] = df["file_path"].progress_apply(
    lambda x: pd.Series(safe_read_shape(x))
)
df["file_path"] = df["file_path"].astype(str)
df.to_parquet("../logs/feature_file_info.parquet", index=False)
# else:
#     df = pd.read_parquet("../logs/feature_file_info.parquet")

100%|██████████| 36832/36832 [01:07<00:00, 548.06it/s]


In [5]:
df.sort_values(["patient_id", "well_fov"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,file_path,patient_id,well_fov,feature_type,compartment,df_shape,missing_values
0,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
1,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cytoplasm,"(13, 18)",0
2,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cytoplasm,"(13, 18)",0
3,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
4,/home/lippincm/Documents/NF1_3D_organoid_profi...,NF0014_T1,C10-1,Granularity,Cell,"(13, 18)",0
...,...,...,...,...,...,...,...
36827,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-6,Granularity,Cell,"(13, 18)",0
36828,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0
36829,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0
36830,/home/lippincm/Documents/NF1_3D_organoid_profi...,SARCO361_T1,G9-7,Granularity,Cytoplasm,"(18, 18)",0


In [6]:
# merge the cells, cytoplasm, and whole cell features for a given well_fov and patient_id
# check for missing values and shape of the dataframes
out_dict = {
    "patient_id": [],
    "well_fov": [],
    "path": [],
    "type": [],
    "feature_type": [],
}
for row in tqdm(
    df.itertuples(), total=df.shape[0], desc="Merging features", leave=True
):
    out_dict["patient_id"].append(row.patient_id)
    out_dict["well_fov"].append(row.well_fov)
    out_dict["path"].append(row.file_path)
    out_dict["type"].append(f"{row.compartment}")
    out_dict["feature_type"].append(row.feature_type)
out_df = pd.DataFrame(out_dict)
out_df.drop_duplicates(subset=["patient_id", "well_fov", "type"], inplace=True)
# pivot such that each type has its own column
out_df = out_df.pivot(
    index=[
        "patient_id",
        "well_fov",
    ],
    columns="type",
    values="path",
).reset_index()

Merging features: 100%|██████████| 36832/36832 [00:00<00:00, 718222.37it/s]


In [7]:
out_df

type,patient_id,well_fov,Cell,Cytoplasm,Nuclei
0,NF0014_T1,C10-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
1,NF0014_T1,C10-2,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
2,NF0014_T1,C11-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3,NF0014_T1,C11-2,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
4,NF0014_T1,C2-1,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
...,...,...,...,...,...
3346,SARCO361_T1,G9-3,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3347,SARCO361_T1,G9-4,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...
3348,SARCO361_T1,G9-5,NaN,/home/lippincm/Documents/NF1_3D_organoid_profi...,NaN
3349,SARCO361_T1,G9-6,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...,/home/lippincm/Documents/NF1_3D_organoid_profi...


In [8]:
labels_dict = {
    "Cell_labels": [],
    "Cytoplasm_labels": [],
    "Nuclei_labels": [],
    "patient_id": [],
    "well_fov": [],
}

# merge the dataframes and check for missing values and shape
for row in tqdm(
    out_df.itertuples(),
    total=out_df.shape[0],
    desc="Checking merged features",
    leave=True,
):
    try:
        cell_df = pd.read_parquet(row.Cell)
        cytoplasm_df = pd.read_parquet(row.Cytoplasm)
        nuclei_df = pd.read_parquet(row.Nuclei)
        labels_dict["Cell_labels"].append(cell_df["object_id"].tolist())
        labels_dict["Cytoplasm_labels"].append(cytoplasm_df["object_id"].tolist())
        labels_dict["Nuclei_labels"].append(nuclei_df["object_id"].tolist())
        labels_dict["patient_id"].append(row.patient_id)
        labels_dict["well_fov"].append(row.well_fov)
    except Exception as e:
        print(f"Error reading files for {row.patient_id} {row.well_fov}: {e}")
labels_df = pd.DataFrame(labels_dict)

Checking merged features:  15%|█▌        | 513/3351 [00:02<00:11, 239.51it/s]

Error reading files for NF0016_T1 D8-1: cannot construct a FileSource from nan


Checking merged features:  39%|███▊      | 1294/3351 [00:05<00:09, 226.92it/s]

Error reading files for NF0030_T1 G2-1: cannot construct a FileSource from nan
Error reading files for NF0030_T1 G2-2: cannot construct a FileSource from nan
Error reading files for NF0030_T1 G2-3: cannot construct a FileSource from nan


Checking merged features:  41%|████▏     | 1389/3351 [00:05<00:08, 228.42it/s]

...

Error reading files for NF0040_T1 G7-5: cannot construct a FileSource from nan
Error reading files for NF0055_T1 F3-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C10-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C10-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C11-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C11-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C11-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C2-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C2-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C3-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C3-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 C4-3: cannot construct a FileSource from nan
Error reading files for SAR

Checking merged features:  95%|█████████▍| 3179/3351 [00:12<00:00, 692.90it/s]

Error reading files for SARCO219_T2 G11-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G11-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G11-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G2-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G2-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G2-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G3-1: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G3-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G3-3: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G3-4: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G4-2: cannot construct a FileSource from nan
Error reading files for SARCO219_T2 G4-4: cannot construct a FileSource from nan
Error reading files for S

Checking merged features: 100%|██████████| 3351/3351 [00:13<00:00, 257.22it/s]

Error reading files for SARCO361_T1 E4-5: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E4-6: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E4-7: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E5-1: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E5-2: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E5-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E5-4: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E5-6: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E5-7: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E6-2: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E6-3: cannot construct a FileSource from nan
Error reading files for SARCO361_T1 E6-6: cannot construct a FileSource from nan
Error reading files for SARC

In [9]:
labels_df["labels_match"] = labels_df.apply(
    lambda row: row["Cell_labels"] == row["Nuclei_labels"],
    axis=1,
)
labels_df["same_number_of_labels"] = labels_df.apply(
    lambda row: len(row["Cell_labels"]) == len(row["Nuclei_labels"]),
    axis=1,
)
labels_df["unique_labels_across_compartments"] = labels_df.apply(
    lambda row: set(row["Nuclei_labels"]) - set(row["Cytoplasm_labels"]), axis=1
)
labels_df.loc[labels_df["labels_match"] == False]

,Cell_labels,Cytoplasm_labels,Nuclei_labels,patient_id,well_fov,labels_match,same_number_of_labels,unique_labels_across_compartments
113,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1, 2, 3, 4, 5, 6, 7, 8, 9]","[1, 2, 3, 4, 5, 7, 8, 9]",NF0014_T2,C11-5,False,False,{}
117,"[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[257, 514, 771, 1028, 1285, 1542, 1799, 2056, ...","[514, 771, 1028, 1285, 1542, 1799, 2056, 2313,...",NF0014_T2,C2-2,False,False,{}
157,"[1, 2, 3, 5, 6, 7, 8, 9, 10, 11]","[1, 2, 3, 5, 6, 7, 8, 9, 10, 11]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]",NF0014_T2,C7-7,False,False,{4}
163,"[257, 771, 1028, 1285, 1542, 1799, 2056]","[257, 771, 1028, 1285, 1542, 1799, 2056]","[257, 771, 1028, 1285, 1542, 2056]",NF0014_T2,C8-6,False,False,{}
175,"[1, 2, 3, 4, 5, 6, 7]","[1, 2, 3, 4, 5, 6, 7]","[2, 3, 4, 5, 6, 7]",NF0014_T2,D10-4,False,False,{}
182,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]","[1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12]",NF0014_T2,D11-4,False,False,{}
183,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[1, 2, 4, 5, 6, 7, 8, 9, 10]",NF0014_T2,D11-5,False,False,{}
201,"[257, 771, 1028, 1285, 1542, 1799, 2056, 2313,...","[257, 771, 1028, 1285, 1542, 1799, 2056, 2313,...","[257, 771, 1285, 1542, 1799, 2056, 2313, 2570,...",NF0014_T2,D4-2,False,False,{}
202,"[3, 5, 6, 8, 9, 12]","[3, 5, 6, 8, 9, 12]","[3, 5, 6, 9, 12]",NF0014_T2,D4-3,False,False,{}
209,"[257, 514, 771, 1285, 1542, 1799, 2056, 3341, ...","[257, 514, 771, 1285, 1542, 1799, 2056, 3341, ...","[257, 514, 771, 1285, 1542, 1799, 3341, 3855, ...",NF0014_T2,D5-3,False,False,{}


In [10]:
# get the patient_id and well_fov for the rows where the labels do not match and check the corresponding feature files for those rows
mismatched_labels = labels_df.loc[
    labels_df["labels_match"] == False, ["patient_id", "well_fov"]
]
mismatched_labels
for row in tqdm(
    mismatched_labels.itertuples(),
    total=mismatched_labels.shape[0],
    desc="Checking mismatched labels",
    leave=True,
):
    patient_id = row.patient_id
    well_fov = row.well_fov
    print(f"cd ../../{patient_id}/extracted_features/ ; rm -r {well_fov}")

Checking mismatched labels: 100%|██████████| 38/38 [00:00<00:00, 91547.13it/s]

cd ../../NF0014_T2/extracted_features/ ; rm -r C11-5
cd ../../NF0014_T2/extracted_features/ ; rm -r C2-2
cd ../../NF0014_T2/extracted_features/ ; rm -r C7-7
cd ../../NF0014_T2/extracted_features/ ; rm -r C8-6
cd ../../NF0014_T2/extracted_features/ ; rm -r D10-4
cd ../../NF0014_T2/extracted_features/ ; rm -r D11-4
cd ../../NF0014_T2/extracted_features/ ; rm -r D11-5
cd ../../NF0014_T2/extracted_features/ ; rm -r D4-2
cd ../../NF0014_T2/extracted_features/ ; rm -r D4-3
cd ../../NF0014_T2/extracted_features/ ; rm -r D5-3
cd ../../NF0014_T2/extracted_features/ ; rm -r D6-1
cd ../../NF0014_T2/extracted_features/ ; rm -r D6-2
cd ../../NF0014_T2/extracted_features/ ; rm -r D7-2
cd ../../NF0014_T2/extracted_features/ ; rm -r D9-3
cd ../../NF0014_T2/extracted_features/ ; rm -r E11-5
cd ../../NF0014_T2/extracted_features/ ; rm -r E2-4
cd ../../NF0014_T2/extracted_features/ ; rm -r E3-4
cd ../../NF0014_T2/extracted_features/ ; rm -r E6-1
cd ../../NF0014_T2/extracted_features/ ; rm -r E7-3
cd ../.

In [11]:
for patient_id in patients:
    print(
        f'cd ../../{patient_id}/extracted_features/ ; find . -type f -name "*Intensity*" -delete'
    )
    print(
        f'cd ../../{patient_id}/extracted_features/ ; find . -type f -name "*Colocalization*" -delete'
    )
    print(
        f'cd ../../{patient_id}/extracted_features/ ; find . -type f -name "*Granularity*" -delete'
    )
    print(
        f'cd ../../{patient_id}/extracted_features ; find . -type f -name "*Texture*" -delete'
    )
    print(
        f'cd ../../{patient_id}/extracted_features ; find . -type f -name "*Neighbors*" -delete'
    )

cd ../../NF0014_T1/extracted_features/ ; find . -type f -name "*Intensity*" -delete
cd ../../NF0014_T1/extracted_features/ ; find . -type f -name "*Colocalization*" -delete
cd ../../NF0014_T1/extracted_features/ ; find . -type f -name "*Granularity*" -delete
cd ../../NF0014_T1/extracted_features ; find . -type f -name "*Texture*" -delete
cd ../../NF0014_T1/extracted_features ; find . -type f -name "*Neighbors*" -delete
cd ../../NF0014_T2/extracted_features/ ; find . -type f -name "*Intensity*" -delete
cd ../../NF0014_T2/extracted_features/ ; find . -type f -name "*Colocalization*" -delete
cd ../../NF0014_T2/extracted_features/ ; find . -type f -name "*Granularity*" -delete
cd ../../NF0014_T2/extracted_features ; find . -type f -name "*Texture*" -delete
cd ../../NF0014_T2/extracted_features ; find . -type f -name "*Neighbors*" -delete
cd ../../NF0016_T1/extracted_features/ ; find . -type f -name "*Intensity*" -delete
cd ../../NF0016_T1/extracted_features/ ; find . -type f -name "*Coloca

In [12]:
labels_df.loc[labels_df["same_number_of_labels"] == False].value_counts("patient_id")

patient_id
NF0014_T2    28
NF0040_T1     6
NF0035_T1     4
Name: count, dtype: int64

In [13]:
# # show the well fov and the patient id
# # print the unique well fovs that have mismatched labels
# for row in labels_df.loc[labels_df["labels_match"] == False][
#     ["patient_id", "well_fov"]
# ].itertuples():
#     print(f"cd ../../{row.patient_id}/extracted_features/ ; rm -r {row.well_fov}")

In [14]:
tmp_df = pd.merge(
    left=pd.merge(
        left=cell_df,
        right=cytoplasm_df,
        on=["object_id", "image_set"],
    ),
    right=nuclei_df,
    on=["object_id", "image_set"],
)
tmp_df.head()

,image_set,object_id,Cell_AGP_Granularity_1,Cell_AGP_Granularity_2,Cell_AGP_Granularity_3,Cell_AGP_Granularity_4,Cell_AGP_Granularity_5,Cell_AGP_Granularity_6,Cell_AGP_Granularity_7,Cell_AGP_Granularity_8,...,Nuclei_Mito_Granularity_7,Nuclei_Mito_Granularity_8,Nuclei_Mito_Granularity_9,Nuclei_Mito_Granularity_10,Nuclei_Mito_Granularity_11,Nuclei_Mito_Granularity_12,Nuclei_Mito_Granularity_13,Nuclei_Mito_Granularity_14,Nuclei_Mito_Granularity_15,Nuclei_Mito_Granularity_16
0,G9-6,1,16.104172,1.970163,1.529468,6.219732,5.100858,6.722161,8.139821,0.0,...,3.936938,3.967296,3.975250,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
1,G9-6,2,39.611074,14.116968,0.000000,12.159025,6.784916,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
2,G9-6,3,9.137148,0.145719,0.598893,6.369452,7.206556,0.000000,0.000000,0.0,...,0.000000,0.000000,3.507180,0.0,0.0,0.0,3.533530,0.0,0.0,3.564691
3,G9-6,4,2.067450,2.589675,2.903647,10.183098,8.004644,0.000000,0.000000,0.0,...,0.000000,0.000000,3.944703,0.0,0.0,0.0,4.120513,0.0,0.0,4.252967
4,G9-6,5,2.767507,1.335198,1.669861,7.917151,8.020715,0.000000,0.000000,0.0,...,0.000000,0.000000,3.739917,0.0,0.0,0.0,3.766272,0.0,0.0,3.772596
